In [34]:
# --- Python version check ---
import sys
assert sys.version_info >= (3, 10), f"Python >= 3.10 required, got {sys.version}"

# --- Library version checks ---
from packaging.version import Version
import sklearn

assert Version(sklearn.__version__) >= Version("1.6.1"), (
    f"scikit-learn >= 1.6.1 required, got {sklearn.__version__}"
)

# --- Core imports ---
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error


In [14]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
target_col = "MedHouseVal"  # already in the frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [16]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Treat test_df as "hands off" until the very end
train_df.shape, test_df.shape
train_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
14196,3.2596,33.0,5.017657,1.006421,2300.0,3.691814,32.71,-117.03,1.030
8267,3.8125,49.0,4.473545,1.041005,1314.0,1.738095,33.77,-118.16,3.821
17445,4.1563,4.0,5.645833,0.985119,915.0,2.723214,34.66,-120.48,1.726
14265,1.9425,36.0,4.002817,1.033803,1418.0,3.994366,32.69,-117.11,0.934
2271,3.5542,43.0,6.268421,1.134211,874.0,2.300000,36.78,-119.80,0.965


In [23]:
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col].copy()

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col].copy()

In [24]:
# All features are numeric in fetch_california_housing
num_cols = list(X_train.columns)
num_cols


['MedInc',
 'HouseAge',
 'AveRooms',
 'AveBedrms',
 'Population',
 'AveOccup',
 'Latitude',
 'Longitude']

In [25]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

In [37]:
model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

pipe = Pipeline(steps=[
    ("prep", numeric_pipeline),
    ("model", model),
])
print(pipe)

Pipeline(steps=[('prep',
                 Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                 ('scaler', StandardScaler())])),
                ('model',
                 RandomForestRegressor(n_estimators=300, n_jobs=-1,
                                       random_state=42))])


In [27]:
scores = cross_val_score(
    pipe,
    X_train, y_train,
    scoring="neg_root_mean_squared_error",
    cv=5
)

rmse = -scores
rmse.mean(), rmse.std()

(np.float64(0.5098597773764081), np.float64(0.005037524516626648))

In [36]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
test_rmse = root_mean_squared_error(y_test, y_pred)
test_rmse

0.5032284255711813